# Task 1: Generating Data

For building the **positive-negative data pairs**, the **negative samples** were generated using a pretrained **NanoGPT** model, providing a diverse set of incorrect answers. The **positive samples**, which require accurate and human-preference style answers, were generated **programmatically** rather than relying on **ChatGPT**, due to observed inaccuracies in its responses for this task.

We designed **12 distinct arithmetic problem cases** to cover a broad range of basic math operations and equation forms:

1. Addition: `a + b = ?`
2. Subtraction: `a - b = ?`
3. Multiplication: `a * b = ?`
4. Division: `a / b = ?` (only exact divisions)
5. Addition with unknown $x$ _(version 1)_: `x + a = b, x = ?`
6. Addition with unknown $x$ _(version 2)_: `a + x = b, x = ?`
7. Subtraction with unknown $x$ _(version 1)_: `a - x = b, x = ?`
8. Subtraction with unknown $x$ _(version 2)_: `x - a = b, x = ?`
9. Multiplication with unknown $x$ _(version 1)_: `a * x = b, x = ?` (only exact divisions)
10. Multiplication with unknown $x$ _(version 2)_: `x * a = b, x = ?` (only exact divisions)
11. Division with unknown $x$ _(version 1)_: `a / x = b, x = ?` (only exact divisions)
12. Division with unknown $x$ _(version 2)_: `x / a = b, x = ?`

For each case, **positive examples** were generated with correct answers and explanations, while **negative examples** were created by appending random incorrect phrases sampled from the NanoGPT-generated negatives.

After generating all samples, we **categorized them based on the actual mathematical operation** involved in solving the problem, rather than just the surface form of the question. For example, even if the question is presented as `$a * x = b, x = ?$`, the calculation to find $x$ involves division (`$x = b / a$`). This approach ensures that the dataset is balanced according to the true arithmetic operation required to solve each problem: **addition, subtraction, multiplication, or division**.

To maintain **balanced representation** and avoid bias towards any particular operation, we adjusted the dataset so that each operation had an equal number of samples—specifically, **25,000 per operation**. This was done by either randomly sampling down from larger sets or duplicating samples when the count was insufficient. The final balanced dataset contains approximately **100,000 positive-negative pairs**, which provides a sufficiently large and diverse training set to support robust model performance.

In [1]:
# run to show first few pairs of data generated

# Task 2

## Step 1: Install necessary packages

In [2]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


# Step 2: Package imports and configuration


In [3]:
# import libraries

import sys
import os
sys.path.append(os.path.abspath(".."))
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
from model import GPT, GPTConfig

import torch

# Check if GPU is available
print("CUDA available:", torch.cuda.is_available())

# Check which device is being used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Optional: print GPU name
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
Using device: cuda
GPU name: NVIDIA GeForce RTX 3060 Ti


Config explanations:
- Beta was reduced to 0.30 here, making the model more conservative, strtaying closer to the original pretrained model during training
- base_lr refers to the size of the step a model has to take adjust parameters during training. It is smaller than original, hence giving us a more stable training process while still allowinng for decent training speed
- Weight decay penalizes large weights of the model, and we found 0.01 to help prevent overfitting
-  We lowered the temperarure from original 0.8 to 1e-8. Temperature controls the randomness of generated text, so a temperature close to 0 helped make the model nearly deterministic
- Top_k  is used to truncate the smapling distribution to top k most likely tokens before sampling. Hence we reduced from the original to reduce randomness of output  
- The other configs were kept the same as changes to them were found to have ill effects on model output

In [4]:
# Configuration
# After fine-tuning to ensure model accuracy

beta = 0.30
base_lr = 9e-5
weight_decay = 0.01
epochs = 5
batch_size = 64
max_length = 64
num_samples = 1
max_new_tokens = 200
temperature = 1e-8
top_k = 20


In [5]:
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

In [6]:
# run to see allowed characters
print(stoi)

{'\n': 0, ' ': 1, "'": 2, '*': 3, '+': 4, ',': 5, '-': 6, '.': 7, '/': 8, '=': 9, '?': 10, '’': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, 'A': 22, 'B': 23, 'C': 24, 'D': 25, 'E': 26, 'F': 27, 'G': 28, 'H': 29, 'I': 30, 'J': 31, 'K': 32, 'L': 33, 'M': 34, 'N': 35, 'O': 36, 'P': 37, 'Q': 38, 'R': 39, 'S': 40, 'T': 41, 'U': 42, 'V': 43, 'W': 44, 'X': 45, 'Y': 46, 'Z': 47, 'a': 48, 'b': 49, 'c': 50, 'd': 51, 'e': 52, 'f': 53, 'g': 54, 'h': 55, 'i': 56, 'j': 57, 'k': 58, 'l': 59, 'm': 60, 'n': 61, 'o': 62, 'p': 63, 'q': 64, 'r': 65, 's': 66, 't': 67, 'u': 68, 'v': 69, 'w': 70, 'x': 71, 'y': 72, 'z': 73}


# Step 3: Define helper functions

In [7]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

# Step 4: Load the pretrained NanoGPT model

In [8]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

# Step 5: Load Data (students are required to complete this part!)

In [9]:
import json
import tiktoken
lines = ""
with open("../pos_neg_pairs.json", "r", encoding = "utf-8") as f:
    lines = json.load(f)
    print(f"Loaded {len(lines)} pairs.")

Loaded 100000 pairs.


The tokenizer metadata file `meta.pkl` contains important mappings used for text encoding and decoding. Specifically, it includes:

* The `stoi` dictionary, which maps each character to a unique token ID.
* The `itos` dictionary, which maps token IDs back to characters.

By inspecting the `stoi` dictionary, we can see the exact set of characters that the tokenizer recognizes and can process. This set includes digits, letters (both uppercase and lowercase), common punctuation marks, and some special characters relevant to our math problem dataset.

In [10]:
print(stoi)

{'\n': 0, ' ': 1, "'": 2, '*': 3, '+': 4, ',': 5, '-': 6, '.': 7, '/': 8, '=': 9, '?': 10, '’': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, 'A': 22, 'B': 23, 'C': 24, 'D': 25, 'E': 26, 'F': 27, 'G': 28, 'H': 29, 'I': 30, 'J': 31, 'K': 32, 'L': 33, 'M': 34, 'N': 35, 'O': 36, 'P': 37, 'Q': 38, 'R': 39, 'S': 40, 'T': 41, 'U': 42, 'V': 43, 'W': 44, 'X': 45, 'Y': 46, 'Z': 47, 'a': 48, 'b': 49, 'c': 50, 'd': 51, 'e': 52, 'f': 53, 'g': 54, 'h': 55, 'i': 56, 'j': 57, 'k': 58, 'l': 59, 'm': 60, 'n': 61, 'o': 62, 'p': 63, 'q': 64, 'r': 65, 's': 66, 't': 67, 'u': 68, 'v': 69, 'w': 70, 'x': 71, 'y': 72, 'z': 73}


These characters define the only allowed characters in our dataset for tokenization. Any character outside this set cannot be properly encoded by the tokenizer and would cause issues during model training.

However, some negative examples generated by NanoGPT contain characters that are not in this allowed set. These include strange Unicode artifacts and other characters that do not belong to the tokenizer’s vocabulary.

In [11]:
# print example of weird negative samples with unexpected unicode

Because of this, we implemented a cleaning step to remove any characters not present in the tokenizer’s vocabulary (`stoi` keys). This cleaning ensures that all text data, in both positive and negative samples, compatible with the tokenizer.

This process results in a filtered and standardized dataset* that contains only **valid characters**, making it ready for **tokenization** and subsequent **model training**.

In [12]:
# implement the cleaning function
def clean_text(s, allowed_chars):
    """
    Iterate over each character in the text, keeping only those that are present in the tokenizer's vocabulary
    """
    return ''.join([c for c in s if c in allowed_chars])

allowed_chars = set(stoi.keys())
cleaned_lines = []

for data_pair in lines:
    # apply cleaning function to every record in the dataset
    cleaned_neg_response = clean_text(data_pair['negative'], allowed_chars)

    # even though only negative responses have characters not present in the tokenizer's vocabulary
    # clean the positive responses as well (just to be safe)
    cleaned_pos_response = clean_text(data_pair['positive'], allowed_chars)

    cleaned_lines.append({'negative': cleaned_neg_response, 'positive': cleaned_pos_response})

lines = cleaned_lines

print(f"Dataset has been cleaned!")

lines = cleaned_lines

Dataset has been cleaned!


In [13]:
# check the same lines we identified earlier to have the unicode characters, and see the difference

# Step 6: Build the optimizer and scheduler (students are required to complete this part!)

## In this step, we build our optimizer and scheduler.

### *Optimizer*
What is an optimizer?
*   Updates the models weights to make the loss go down as the model is training.
*   Uses gradients from backprop **`.backward()`**  to determine the change to be made to the weights

In the model.py file, there is a helper function **configure_optimizers**.
What **`configure_optimizers`** does:
1. Collects all the trainable model parameters
2. Splits the parameters into `decay_params` and `nodecay_params` based on tensor dimensionality `dim`
3. Applies the weight decay to `decay_params` only
4. Builds the AdamW optimizer and returns it for use

We hence called the function to build our optimizer.

### *Scheduler*
What is a scheduler?
*   Controls the learning rate over time as the the model trains
*   It decides how aggressive each update should be by adjusting the optimizer's learning rate

Why do we need a scheduler?
*   The learning rate rate affects the speed and accuracy of training.
*   If learning rate is too high, it results in the update to the weights to be large. This results in the loss to spike or keep fluctuating instead of decreasing.
*   If its too low, the model technically improved, but the training might become too slow or stuck in a sub optimal region as the steps are too small

For our NanoGPT, our group used a **cosine annealing scheduler with warmup**.
How it works:
*   During warm up, learning rate (`lr`) starts low and increases til it reaches the target peak learning rate, when `max_lr = base_lr`. This helps to avoid instability at the start of training.
*   Once the warm up is complete, the learning rate will slowly decrease to a minimum, `min_lr = 0.1 * base_lr`. This will ensure that the updates are more careful as the model trains.
*   This gradual decrease follows a cosine curve, resulting in the learning rate to decay slowly at the beginning, then becomes more aggressive near the end.

In [14]:
import math
optimizer = gpt.configure_optimizers(weight_decay, base_lr, (0.9, 0.95), device)

def scheduler(total_steps, base_lr, current_step):

    # peak learning rate that scheduler will reach after warming up, from which cosine decay will start
    max_lr = base_lr

    # lowest learning rate that scheduler will reach during cosine decay
    min_lr = 0.1 * base_lr

    # number of initial iterations during which learning rate linearly increases
    warmup_steps = round(0.03 * total_steps)

    # 1. warmup phase
    if current_step <= warmup_steps:
        lr = max_lr * (current_step / warmup_steps)
    # 2. post decay phase
    elif current_step >= total_steps:
        lr = min_lr
    # 3. cosine annealing phase
    else:
        p = (current_step - warmup_steps) / (total_steps - warmup_steps)
        lr = min_lr + 0.5*(max_lr - min_lr)*(1 + math.cos(math.pi * p))

    return lr

num decayed parameter tensors: 26, with 8,834,328 parameters
num non-decayed parameter tensors: 13, with 4,524 parameters
using fused AdamW: False


# Step 7: Begin training (students are required to complete this part!)

To optimize training speed and efficiency on GPUs, we enable **Automatic Mixed Precision (AMP)** with **float16 (fp16)** precision:

- If running on a **GPU**, compute-heavy operations (like matrix multiplications) use fp16 on Tensor Cores for faster execution.
- If running on a **CPU**, AMP is automatically disabled and computations run in standard float32 for stability.

Since our dataset is moderate in size (~100k samples), this setup balances speed and numerical stability effectively.

In [15]:
# AMP will be set to True only if running on a CUDA GPU
use_amp = torch.cuda.is_available()

# use fp16 on GPU (ignored on CPU)
autocast_dtype = torch.float16

# enable GradScaler only if AMP is active, i.e. if running on a CUDA GPU (ignored on CPU)
scaler = torch.amp.GradScaler(enabled=use_amp)

# set up autocast context manager to automatically cast operations to fp16 where beneficial (perform mixed precision operations)
# this will be enabled only if AMP is active, i.e. if running on a CUDA GPU (ignored on CPU)
ctx = torch.amp.autocast(device_type='cuda', dtype=autocast_dtype, enabled=use_amp)

print(f"AMP enabled: {use_amp}, dtype: {autocast_dtype}, using GradScaler: {scaler.is_enabled()}")

AMP enabled: True, dtype: torch.float16, using GradScaler: True



We can split the training loop into 2 main parts: **Forward Pass** and **Backward Pass**


1. **Forward Pass**

The main purpose of the forward pass is to compute the model outputs and the loss for the current batch.

**How we computed the loss:** <br/>

Our loss is calculated based on this equation:<br/>
`loss = -F.logsigmoid((pos_log - neg_log) / beta).mean() - 0.1 * pos_log.mean()`

It's composed of two key parts:

* Comparison: `-F.logsigmoid((pos_log - neg_log) / beta).mean()`
    * This term encourages the model to assign a higher score to positive samples (`pos_log`) than to negative samples (`neg_log`)
    * It helps in penalising the model when there is no clear margin between the positive answer and the negative answer

* Regularisation: `- 0.1 * pos_log.mean()`
    * This applies an additional small (`0.1`) penalty to encourage the model to assign high probabilities to the positive samples.

2. **Backward Pass**

The main purpose of the backward pass is to computes gradients of the loss w.r.t. all model parameters and update them using the optimizer.

However, on GPUs where we are using fp16, we have a problem: fp16 numbers have limited precision, hence gradients can become too small to represent (underflow) and get lost, which would prevent the model from learning.

- To solve this problem, we use a **GradScaler** to prevent gradient underflow in fp16 by dynamically scaling the loss before backpropagation (calculating the gradients) and unscaling gradients before the optimizer step (updating the model with the new gradients)

In [16]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor, pos_tensor) in enumerate(pbar):

        # training loop:

        # clear gradients at the start of the batch to avoid accumulation from previous batch
        # using set_to_none=True also frees memory by releasing old gradient tensors
        optimizer.zero_grad(set_to_none=True)

        # 1. forward pass:

        # on a GPU: use mixed precision for speed
        # -  operations will use fp16 automatically via autocast
        # vs on a CPU: autocast does nothing, forward pass runs in float32
        with ctx:
            neg_log = compute_logprob(neg_tensor) # compute log-probabilities for negative samples
            pos_log = compute_logprob(pos_tensor) # compute log-probabilities for positive samples

            # compute combined loss with regularisation
            loss = -F.logsigmoid((pos_log - neg_log) / beta).mean() - 0.1 * pos_log.mean()

        # update global step: used to track how many batches have been processed in total so far (across all epochs)
        global_step = epoch * total_steps + step + 1

        # calculate the learning rate for the current step
        lr = scheduler(total_steps * epochs, base_lr, global_step)
        # update all parameter groups to the current step learning rate computed by the scheduler
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # 2. backward pass:

        # on a GPU:
        if scaler.is_enabled():
            # first scale the loss to prevent gradient underflow
            # then compute the gradients (in float16)
            scaler.scale(loss).backward()

            # unscale gradients before clipping so that gradient norms are correct
            scaler.unscale_(optimizer)

            # clip gradients to prevent exploding gradients (ensure stability)
            torch.nn.utils.clip_grad_norm_(gpt.parameters(), max_norm=1.0)

            # update model parameters based on scaled gradients
            scaler.step(optimizer)

            # update GradScaler itself, by increasing / decreasing the scaling factor dynamically
            # - this ensures that scaling is optimal for the next iteration
            scaler.update()

        # vs on a CPU:
        # - we are not using GradScaler because float32 has enough precision (hence gradient underflow is not a concern)
        else:

            # compute the gradients (in float32)
            loss.backward()

            # clip gradients to prevent exploding gradients (ensure stability)
            torch.nn.utils.clip_grad_norm_(gpt.parameters(), max_norm=1.0)

            # update model parameters
            optimizer.step()

        # update progress bar with the loss and learning rate of the current step
        # (to track progress across all batches and epochs)
        pbar.set_description(f"epoch {epoch+1} step {step+1} train={loss.item():.4f} lr={lr:.2e}")

    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

epoch 1 step 1562 train=0.0280 lr=8.40e-05: : 1562it [01:37, 16.03it/s]


Saved checkpoint to ./dpo.pt


epoch 2 step 1562 train=0.0240 lr=6.42e-05: : 1562it [01:35, 16.38it/s]


Saved checkpoint to ./dpo.pt


epoch 3 step 1562 train=0.0223 lr=3.85e-05: : 1562it [01:32, 16.84it/s]


Saved checkpoint to ./dpo.pt


epoch 4 step 1562 train=0.0201 lr=1.72e-05: : 1562it [01:31, 16.99it/s]


Saved checkpoint to ./dpo.pt


epoch 5 step 1562 train=0.0199 lr=9.00e-06: : 1562it [01:32, 16.86it/s]

Saved checkpoint to ./dpo.pt


# Step 8: Begin testing (students are required to complete this part!)

For testing, we generated 3 different cases of inputs to test how our handle models them.
1.   Direct Arithmetic Questions
2.   Solving for x (x in front)
3.   Solving for x (x after operation)

How the testing works:
*   Tokenises each prompt
*   We use `gpt.generate` to feed the prompt into the model to generate an answer
*   We then decode the tokens to convert it back to text

In [17]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()

test_set = [
    "12+9=?",
    "33-11=?",
    "9*7=?",
    "64/8=?",
    "99-83=?",
    "25+79=?",
    "5*18=?",
    "60/12=?",
    "2*18=?",
    "45+69=?",

    # x equations
    "x+12=28,x=?",
    "x-9=37,x=?",
    "x*8=64,x=?",
    "x/4=11,x=?",
    "x+17=45,x=?",
    "x-11=90,x=?",
    "x*9=81,x=?",
    "x/2=14,x=?",
    "x*7=84,x=?",
    "x+33=99,x=?",

    # reverse-style equations
    "45+x=90,x=?",
    "72-x=63,x=?",
    "18/x=3,x=?",
    "7*x=28,x=?",
    "81/x=9,x=?",
    "6*x=54,x=?",
    "13+x=27,x=?",
    "99-x=1,x=?",
    "48/x=2,x=?",
    "20/x=4,x=?"
]

with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################

        # convert the prompts which were encoded into tensor
        prompt_ids_formatted = torch.tensor([prompt_ids], dtype=torch.long, device=device)

        # generates result
        result = gpt.generate(prompt_ids_formatted, max_new_tokens, temperature, top_k)

        # process the generated tokens back into readable text
        decoded_result = decode(result[0].detach().cpu().view(-1).tolist())

        print(decoded_result)
        ###########################################################

12+9=? The answer is 21 because 12+9 equals 21.
33-11=? The answer is 22 because 33-11 equals 22.
9*7=? The answer is 63 because 9*7 equals 63.
64/8=? The answer is 8 because 64/8 equals 8.
99-83=? The answer is 16 because 99-83 equals 16.
25+79=? The answer is 104 because 25+79 equals 104.
5*18=? The answer is 90 because 5*18 equals 90.
60/12=? The answer is 5 because 60/12 equals 5.
2*18=? The answer is 36 because 2*18 equals 36.
45+69=? The answer is 114 because 45+69 equals 114.
x+12=28,x=? The answer is 16 because 28-12 equals 16.
x-9=37,x=? The answer is 46 because 37+9 equals 46.
x*8=64,x=? The answer is 8 because 64/8 equals 8.
x/4=11,x=? The answer is 44 because 4*11 equals 44.
x+17=45,x=? The answer is 28 because 45-17 equals 28.
x-11=90,x=? The answer is 101 because 90+11 equals 101.
x*9=81,x=? The answer is 9 because 81/9 equals 9.
x/2=14,x=? The answer is 32 because 2*14 equals 32.
x*7=84,x=? The answer is 12 because 84/7 equals 12.
x+33=99,x=? The answer is 66 because 99-